Copyright 2025 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left"> <td>      <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Demos/Emoji-Gemma-on-Web/resources/Fine_tune_Gemma_3_270M_for_emoji_generation.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

# 微調 Gemma 3 270M 用於表情符號生成

notebook 透過 Hugging Face 變壓器強化學習 ([TRL](https://huggingface.co/docs/trl/en/index)) 對 Gemma 進行微調，以使用量化低階適應 (QLoRA) 將文本翻譯成表情符號的任務 @@P004404004 記憶體的速度@P004 階段 @@P004。
在 Google Colab T4 GPU 加速器上訓練 [Gemma 3 270M](https://huggingface.co/google/gemma-3-270m) 時，此過程端到端只需 10 分鐘。執行每個程式碼片段以：
1. 設定Colab環境
2. 為fine-tuning準備一個dataset
3. 加載並測試底座Gemma 3 270M 模型
4. 微調模型
5. 測試、評估並保存模型以供進一步使用

## 設定開發環境

第一步是使用 `pip` 軟體包安裝程式安裝必要的庫。

In [ ]:
%pip install torch tensorboard emoji
%pip install -U transformers trl datasets accelerate evaluate sentencepiece bitsandbytes protobuf==3.20.3

您可以重新啟動會話 (runtime) 以使用新安裝的庫。

##啟用Hugging Face權限
要使用 Gemma 模型，您需要接受模型使用授權並建立存取權杖：
1. **接受[模型頁](http://huggingface.co/google/gemma-3-270m-it) 上的許可證**。

2. **取得具有「寫入」存取權限的有效[存取權杖](https://huggingface.co/settings/tokens)（非常重要！）**

3. 在左側工具列中建立新的Colabsecret。指定`HF_TOKEN` 作為“名稱”，添加您唯一的token 作為“值”，然後開啟“notebook 訪問”。

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Login into Hugging Face Hub
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

## 加載dataset

Hugging Face 託管大量 dataset 集合，用於訓練和評估模型。如果您不使用自訂dataset，則可以載入包含文字範例和對應表情符號翻譯的[預製dataset](https://huggingface.co/datasets/kr15t3n/g-emoji)。
**如果您想使用自己的自訂dataset，請跳至下一步。 **

In [ ]:
import emoji
from datasets import load_dataset

# Use the first 2000 samples for efficient training.
general_dataset_path = load_dataset("kr15t3n/text2emoji", encoding="utf-8", split="train")

# Clean dataset to only use examples where 'emoji' field contains only emoji characters
def is_only_emoji(sample):
  emoji_string = sample['emoji']
  if not emoji_string:
    return False
  return all(emoji.is_emoji(char) for char in emoji_string)
dataset = general_dataset_path.filter(is_only_emoji)

print(f"\nHere's the 10th example from the dataset: {dataset[10]}")

### 上傳自訂dataset
**如果您已經載入了dataset，請跳過此步驟。 **

您可以透過建立包含以鍵值對形式建構的文字到表情符號 dataset 的電子表格來自訂 Gemma 3 270M 以使用特定表情符號。如果您想鼓勵記住特定表情符號，我們建議提供 10-20 個具有不同文字變體的表情符號範例。
使用[預製dataset](https://github.com/google-gemini/gemma-cookbook/blob/main/Demos/Emoji-Gemma-on-Web/resources/Emoji%20Translation%20Dataset%20-%20Dataset.csv)作為範本建立自己的dataset，然後將其上傳到左側工具列中的 Files 資料夾中。透過右鍵單擊該檔案並在 `custom_dataset_path` 中指向它來獲取其路徑。

In [ ]:
import emoji
from datasets import load_dataset
from transformers import AutoTokenizer

# Point to your uploaded dataset
custom_dataset_path = "/content/Emoji Translation Dataset - Dataset.csv"      #@param {type:"string"}
dataset = load_dataset("csv", data_files=custom_dataset_path, encoding="utf-8", split="train")

# Clean dataset to only use examples where 'emoji' field contains only emoji characters
def is_only_emoji(sample):
  emoji_string = sample['emoji']
  if not emoji_string:
    return False
  return emoji.purely_emoji(emoji_string)
dataset = dataset.filter(is_only_emoji)

print(f"\nHere's the 10th example from your dataset: {dataset[10]}")

## 載入模型

您可以透過接受授權條款從Hugging Face Hub存取[Gemma 3 270M](https://huggingface.co/google/gemma-3-270m-it)。該模型的指令調整版本已經接受瞭如何遵循指示的培訓，並且透過fine-tuning，您現在將使其適應新任務。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

gemma_model = "google/gemma-3-270m-it"
base_model = AutoModelForCausalLM.from_pretrained(gemma_model, device_map="auto", attn_implementation="eager", dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(gemma_model)

print(f"Device: {base_model.device}")
print(f"DType: {base_model.dtype}")

如果您使用的是 GPU runtime，裝置應列印為 `cuda`。 **如果您還沒準備好，請在 Colab 中使用免費的 T4 GPU runtime 以獲得更快的 fine-tuning。 **

### 格式化訓練dataset
現在您已載入數據，將訓練 dataset 格式化為對話角色，包括文字輸入和表情符號輸出，以及包含模型方向的 system prompt。這有助於模型學習如何解釋 dataset 中的「文字」和「表情符號」列。

In [ ]:
from transformers import AutoTokenizer

def translate(sample):
  return {
      "messages": [
          {"role": "system", "content": "Translate this text to emoji: "},
          {"role": "user", "content": f"{sample['text']}"},
          {"role": "assistant", "content": f"{sample['emoji']}"}
      ]
  }

training_dataset = dataset.map(translate, remove_columns=dataset.features.keys())
training_dataset_splits = training_dataset.train_test_split(test_size=0.1, shuffle=True)

print("\nHere's the 40th example from the formatted training dataset:")
print(training_dataset[40])

### 推薦：測試基本模型

我們首先檢查基礎模型如何回應指令“將此文字翻譯為表情符號”
嘗試測試幾次。

In [ ]:
from transformers import pipeline
from random import randint
import re

# Create a transformers inference pipeline
pipe = pipeline("text-generation", model=gemma_model, tokenizer=tokenizer)

# Select a random sample from the test dataset
rand_idx = randint(0, len(training_dataset_splits["test"]) - 1)
test_sample = training_dataset_splits["test"][rand_idx]

# Handle messages
all_messages = test_sample['messages']
user_message_content = next((msg['content'].strip() for msg in all_messages if msg['role'] == 'user'), "Not Found")
dataset_emoji = next((msg['content'].strip() for msg in all_messages if msg['role'] == 'assistant'), "Not Found")
prompt_messages = [
    {"role": "system", "content": "Translate this text to emoji: "},
    {"role": "user", "content": user_message_content}
]

# Apply the chat template. This will format the messages correctly for the model.
prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)

# Generate the output
output = pipe(prompt, max_new_tokens=64)
model_output_only = output[0]['generated_text'][len(prompt):].strip()

print(f"\nDataset text: {user_message_content}")
print(f"\nDataset emoji: {dataset_emoji}")
print(f"\nModel generated output: {model_output_only}")

基本模型輸出可能無法滿足您的期望 - 但這沒關係！
Gemma 3 270M 是為任務專業化而設計的，這意味著當使用代表性範例進行訓練時，它可以提高特定任務的表現。讓我們微調模型以獲得更可靠的輸出。

## 微調模型

Hugging Face [TRL](https://huggingface.co/docs/trl/index) 提供了訓練工具和fine-tuning LLM，使用QLoRA（量化低階適應）等記憶體高效技術在模型的凍結量化版本之上訓練適配器。

### 設定調優作業
定義 Gemma 3 基本模型的訓練設定：
1. `BitsandBytesConfig` 量化模型以提高記憶體效率
2. `LoraConfig` 用於參數高效 fine-tuning
2. `SFTConfig` 用於受監督 fine-tuning

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig

adapter_path = "/content/myemoji-gemma-adapters"      # Where to save your LoRA adapters
tokenizer = AutoTokenizer.from_pretrained(gemma_model)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",                      # Target all linear layers
    lora_dropout=0.05,                                # Increase to 0.1 to induce overfitting
    bias="none",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"]       # Save the lm_head and embed_tokens as you train the special tokens
)

args = SFTConfig(
    output_dir=adapter_path,                          # Directory to save adapters
    num_train_epochs=3,                               # Number of training epochs
    per_device_train_batch_size=4,                    # Batch size per device during training
    logging_strategy="epoch",                         # Log every epoch
    eval_strategy="epoch",                            # Evaluate loss metrics every epoch
    save_strategy="epoch",                            # Save checkpoint every epoch
    learning_rate=5e-5,                               # Learning rate,
    lr_scheduler_type="constant",                     # Use constant learning rate scheduler
    max_length=256,                                   # Max sequence length for model and packing of the dataset
    gradient_checkpointing=False,                     # Use gradient checkpointing to save memory
    packing=False,                                    # Groups multiple samples in the dataset into a single sequence
    optim="adamw_torch_fused",                        # Use fused adamw optimizer
    report_to="tensorboard",                          # Report metrics to tensorboard
    weight_decay=0.01,                                # Added weight decay for regularization
)

base_model = AutoModelForCausalLM.from_pretrained(gemma_model, quantization_config=bnb_config, device_map="auto", attn_implementation='eager')
base_model.config.pad_token_id = tokenizer.pad_token_id

print("Training configured")

### 開始訓練

`SFTTrainer` token 化 dataset 並使用上一個步驟中的超參數訓練基本模型。
訓練時間會根據一系列因素而變化，例如 dataset 的大小或 epoch 的數量。使用 T4 GPU，1000 個訓練範例大約需要 10 分鐘。如果訓練進展緩慢，請檢查您是否在 Colab 中使用 T4 GPU。

In [ ]:
from trl import SFTConfig, SFTTrainer

# Set training and evaluation datasets
train_dataset = training_dataset_splits['train']
eval_dataset = training_dataset_splits['test']

# Train and save the LoRA adapters
trainer = SFTTrainer(
    model=base_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
)
trainer.train()
trainer.save_model(adapter_path)

print(f"LoRA adapters saved to {adapter_path}")

每個訓練checkpoint（紀元）的LoRA適配器將保存在您的臨時Colab會話儲存中。現在，您可以評估訓練和驗證損失指標，以選擇與模型合併的適配器。

### 繪製訓練結果
要評估模型，您可以使用 Matplotlib 繪製訓練和驗證損失，以視覺化訓練步驟或曆元的這些指標。這有助於監控訓練過程並就超參數或提前停止做出明智的決策。

In [ ]:
import matplotlib.pyplot as plt

# Access the log history
log_history = trainer.state.log_history

# Extract training / validation loss
train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

# Plot the training loss
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

訓練損失衡量模型訓練資料的誤差。驗證損失測量模型以前未見過的單獨 dataset 上的錯誤。監控兩者有助於檢測過度擬合（當模型在訓練資料上表現良好但在未見資料上表現不佳時）。
- 驗證損失>>訓練損失：**過度擬合**
- 驗證損失>訓練損失：**一些過度擬合**
- 驗證損失<訓練損失：**一些欠擬合**
- 驗證損失 << 訓練損失：**欠擬合**

如果您的任務需要記住特定範例，或為給定文字產生特定表情符號，那麼過度擬合可能是有益的。

### 合併適配器

經過訓練後，您可以將 LoRA 適配器與模型合併。您可以透過指定訓練checkpoint資料夾來選擇要合併的適配器，否則它將預設為最後一個時期。* 為了更好的任務泛化，選擇最欠擬合的checkpoint（驗證損失<訓練損失）
* 為了更好地記憶具體範例，請選擇最過度擬合的（checkpoint > 訓練損失）


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

adapter_path = "/content/myemoji-gemma-adapters/"                 # Choose which adapters to merge, otherwise defaults to latest
merged_model_path = "/content/myemoji-gemma-merged/"              # Location of merged model directory

# Load base model and tokenizer
base_model = AutoModelForCausalLM.from_pretrained(gemma_model, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# Load and merge the PEFT adapters onto the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()

# Save the merged model and its tokenizer
model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)

print(f"Model merged and saved to {merged_model_path}. Final model vocabulary size: {model.config.vocab_size}")

### 測試微調後的模型

讓我們將微調後的模型性能與基本模型進行比較！透過更新 `text_to_translate` 測試一些輸入。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Create Transformers inference pipeline
merged_model = AutoModelForCausalLM.from_pretrained(merged_model_path, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(merged_model_path)
pipe = pipeline("text-generation", model=merged_model, tokenizer=tokenizer)
pipe_base = pipeline("text-generation", model=gemma_model, device_map="auto")

# Test a prompt
text_to_translate = "let's go to the beach"  #@param {type:"string"}
inference_messages = [
    {"role": "system", "content": "Translate this text to emoji: "},
    {"role": "user", "content": text_to_translate}
]
prompt = tokenizer.apply_chat_template(inference_messages, tokenize=False, add_generation_prompt=True)
output = pipe(prompt, max_new_tokens=128)
output_base = pipe_base(prompt, max_new_tokens=128)
model_output = output[0]['generated_text'][len(prompt):].strip()
model_output_base = output_base[0]['generated_text'][len(prompt):].strip()

print(f"\nFine-tuned model output: {model_output}")

print(f"\nBase model output: {model_output_base}")

該模型是否輸出您期望的表情符號？
如果您沒有得到想要的結果，您可以嘗試[使用不同的超參數](#scrollTo=-BJFoOdL0y8w) 來訓練模型，或更新您的訓練dataset 以包含更具代表性的範例。
對結果感到滿意後，您可以將模型儲存到Hugging Face Hub。

## 儲存您的模型並上傳至Hugging Face Hub
**您現在擁有了一個客製化的Gemma 3 270M 模型！ 🎉**
將其上傳到 Hugging Face Hub 上的儲存庫，以便您輕鬆共享您的模型或稍後存取它。

In [ ]:
from huggingface_hub import ModelCard, ModelCardData, whoami

#@markdown Name your model
model_name = "myemoji"                            #@param {type:"string"}

username = whoami()['name']
hf_repo_id = f"{username}/{model_name}-gemma-3-270m-it"

repo_url = model.push_to_hub(hf_repo_id, create_repo=True, commit_message="Upload model")
tokenizer.push_to_hub(hf_repo_id)

card_content = f"""
---
base_model: {gemma_model}
tags:
- text-generation
- emoji
- gemma
---
A fine-tuned model based on `{gemma_model}`."""
card = ModelCard(card_content)
card.push_to_hub(hf_repo_id)

print(f"Uploaded to {repo_url}")

## 摘要與後續步驟

這個notebook涵蓋如何有效地微調Gemma 3 270M 以產生表情符號。繼續執行轉換和量化步驟，為設備上部署做好準備。您可以按照以下步驟進行操作：
1.  [轉換以與MediaPipe LLM Inference API一起使用](https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Demos/Emoji-Gemma-on-Web/resources/Convert_Gemma_3_270M_to_LiteRT_for_MediaPipe_LLM_Inference_API.ipynb)
2.  [透過ONNX Runtime轉換為與Transformers.js一起使用](https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Demos/Emoji-Gemma-on-Web/resources/Convert_Gemma_3_270M_to_ONNX.ipynb)